In [11]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from internasus.loaders import conectar_datasus

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 4)

# Conexão DuckDB com as views mapeadas para os parquets de cada sistema
conn = conectar_datasus()

[conectar_datasus] View 'cnes_eq' criada (79 arquivo(s)).
[conectar_datasus] View 'cnes_lt' criada (79 arquivo(s)).
[conectar_datasus] View 'cnes_pf' criada (72 arquivo(s)).
[conectar_datasus] View 'cnes_sr' criada (79 arquivo(s)).
[conectar_datasus] Aviso: nenhum parquet para 'cnes_st' em C:\ProjetosDS\internasus\data\raw\fonte=CNES\uf=SP\**\dataset=ST\*.parquet — pulando.
[conectar_datasus] View 'sia' criada (156 arquivo(s)).
[conectar_datasus] View 'sih' criada (78 arquivo(s)).
[conectar_datasus] View 'ibge_pop' criada (5 arquivo(s)).


## 0. Exploração de Schemas, Qualidade de Dados e Preparação para a Camada Silver

Antes de responder as perguntas de negócio, é preciso entender a granularidade e a qualidade de cada fonte bruta (bronze) — isso é o que define quais transformações a camada silver precisa aplicar. Os achados desta seção (views auxiliares, variável `ANO_REF`) são reutilizados nas seções seguintes.

In [12]:
tabelas = conn.execute("SHOW TABLES").df()["name"].tolist()
print("Views disponíveis:", tabelas)
print()

resumo = []
for t in tabelas:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    ncols = len(conn.execute(f"DESCRIBE {t}").df())
    resumo.append({"view": t, "linhas": n, "colunas": ncols})

display(pd.DataFrame(resumo).sort_values("linhas", ascending=False))

Views disponíveis: ['cnes_eq', 'cnes_lt', 'cnes_pf', 'cnes_sr', 'ibge_pop', 'sia', 'sih']



,view,linhas,colunas
5,sia,524544253,66
2,cnes_pf,93782990,45
0,cnes_eq,17498047,33
6,sih,17139043,119
3,cnes_sr,11815895,37
1,cnes_lt,664564,33
4,ibge_pop,3225,7


### Achado 1 — CNES (EQ/LT/PF/SR) é um retrato mensal repetido; SIA/SIH são fluxos de eventos

As bases CNES são publicadas por **competência** (mês) e cada equipamento/leito/profissional/serviço aparece **repetido em todas as competências em que esteve ativo**. Somar `QT_EXIST` (ou contar linhas) direto na view bruta multiplica o valor pelo número de meses — o exemplo abaixo mostra o mesmo equipamento presente em até ~79 competências consecutivas (toda a série 2020–2026 baixada).

SIA e SIH, ao contrário, são bases de **eventos** (uma linha = um procedimento/uma internação): somar ao longo do tempo faz sentido nelas.

**Transformação necessária na camada silver:** para qualquer pergunta sobre "quantos equipamentos/leitos/profissionais/serviços existem", selecionar uma única competência de referência (aqui, a mais recente) por fonte, em vez de agregar a série toda.

In [13]:
dup = conn.execute("""
    SELECT CNES, CODEQUIP, COUNT(DISTINCT COMPETEN) AS n_competencias
    FROM cnes_eq
    GROUP BY 1, 2
    ORDER BY n_competencias DESC
    LIMIT 5
""").df()
display(dup)

# Views "silver-like": última competência disponível de cada fonte CNES
for view in ["cnes_eq", "cnes_lt", "cnes_pf", "cnes_sr"]:
    conn.execute(f"""
        CREATE OR REPLACE VIEW {view}_atual AS
        SELECT * FROM {view}
        WHERE COMPETEN = (SELECT MAX(COMPETEN) FROM {view})
    """)
    comp = conn.execute(f"SELECT MAX(COMPETEN) FROM {view}").fetchone()[0]
    n = conn.execute(f"SELECT COUNT(*) FROM {view}_atual").fetchone()[0]
    print(f"{view}_atual -> competência de referência {comp}, {n:,} linhas")

# MICR_REG parou de ser preenchido a partir de 202312 (ver Achado 3) — para análises
# por microrregião (seções 2.4 e 4.4), fixamos a última competência em que o campo
# ainda vinha populado, em vez da competência mais recente.
COMPETEN_MICRORREGIAO = "202311"
for view in ["cnes_lt", "cnes_pf"]:
    conn.execute(f"""
        CREATE OR REPLACE VIEW {view}_micro AS
        SELECT * FROM {view} WHERE COMPETEN = '{COMPETEN_MICRORREGIAO}'
    """)
    n = conn.execute(f"SELECT COUNT(*) FROM {view}_micro").fetchone()[0]
    print(f"{view}_micro -> competência fixa {COMPETEN_MICRORREGIAO} (última com MICR_REG preenchido), {n:,} linhas")

,CNES,CODEQUIP,n_competencias
0,2093200,72,79
1,7298293,13,79
2,3730751,02,79
3,9812806,81,79
4,9087079,46,79


cnes_eq_atual -> competência de referência 202607, 248,040 linhas
cnes_lt_atual -> competência de referência 202607, 8,327 linhas
cnes_pf_atual -> competência de referência 202607, 1,568,324 linhas
cnes_sr_atual -> competência de referência 202607, 182,660 linhas
cnes_lt_micro -> competência fixa 202311 (última com MICR_REG preenchido), 8,375 linhas
cnes_pf_micro -> competência fixa 202311 (última com MICR_REG preenchido), 1,326,844 linhas


### Achado 2 — código de município não bate entre DATASUS (6 dígitos) e IBGE/SIDRA (7 dígitos)

CNES (`CODUFMUN`), SIH (`MUNIC_RES`/`MUNIC_MOV`) e SIA (`PA_MUNPCN`/`PA_UFMUN`) usam o código IBGE **de 6 dígitos** (sem o dígito verificador). A view `ibge_pop` (SIDRA) usa o código **de 7 dígitos**. É por isso que o `JOIN` direto usado na primeira versão do notebook não retornava nada — as chaves nunca coincidiam.

SIH/SIA também trazem município de **outros estados** (pacientes vindos de fora de SP) — esperado e relevante para o bloco 2 (fuga/pressão assistencial), mas gera `NULL` ao cruzar com `ibge_pop` (só tem municípios de SP).

**Transformação necessária na camada silver:** normalizar todos os códigos de município para 6 dígitos (`LEFT(codigo, 6)`) antes de qualquer `JOIN` geográfico.

Além disso, a população do SIDRA tem uma linha por **município × ano**: cruzar sem filtrar o ano primeiro duplica cada município por linha (aqui, 5 anos disponíveis → registros x5).

In [14]:
conn.execute("""
    CREATE OR REPLACE VIEW ibge_pop_sp AS
    SELECT
        LEFT(municipio_codigo, 6) AS cod_mun,
        municipio_nome AS nome_mun,
        ano,
        valor AS populacao
    FROM ibge_pop
""")

anos_pop = conn.execute("SELECT DISTINCT ano FROM ibge_pop_sp ORDER BY 1").df()["ano"].tolist()
print("Anos com estimativa de população disponível:", anos_pop, "-> faltam 2022/2023 (gap do SIDRA, não do pipeline)")

anos_sia_completos = conn.execute("""
    SELECT ano FROM sia GROUP BY ano HAVING COUNT(DISTINCT mes) = 12 ORDER BY ano DESC
""").df()["ano"].tolist()

ANO_REF = max(a for a in anos_pop if a in anos_sia_completos)
print(f"Ano de referência escolhido para as análises per capita: {ANO_REF} "
      "(último ano com população estimada E com os 12 meses de SIA/SIH completos)")

conn.execute(f"""
    CREATE OR REPLACE VIEW ibge_pop_ref AS
    SELECT cod_mun, nome_mun, populacao FROM ibge_pop_sp WHERE ano = {ANO_REF}
""")
display(conn.execute("SELECT * FROM ibge_pop_ref LIMIT 5").df())

Anos com estimativa de população disponível: [2020, 2021, 2024, 2025, 2026] -> faltam 2022/2023 (gap do SIDRA, não do pipeline)
Ano de referência escolhido para as análises per capita: 2025 (último ano com população estimada E com os 12 meses de SIA/SIH completos)


,cod_mun,nome_mun,populacao
0,350010,Adamantina - SP,35673
1,350020,Adolfo - SP,4505
2,350030,Aguaí - SP,32886
3,350040,Águas da Prata - SP,7463
4,350050,Águas de Lindóia - SP,18257


### Achado 3 — como interpretar os códigos de procedimento, equipamento e leito

- `PA_PROC_ID` (SIA) e `PROC_REA`/`PROC_SOLIC` (SIH) são códigos SIGTAP de 10 dígitos; os 2 primeiros indicam o grupo do procedimento — **"02" = finalidade diagnóstica, "04" = procedimentos cirúrgicos** — estável o suficiente para usar diretamente.
- `PA_NIVCPL` (SIA) já vem com o **nível de complexidade** do procedimento (0/1/2/3 ≈ não se aplica / atenção básica / média / alta complexidade) — mais direto do que inferir complexidade pelo grupo SIGTAP.
- Já `TIPEQUIP`/`CODEQUIP` (CNES-EQ) e `TP_LEITO`/`CODLEITO` (CNES-LT) seguem tabelas de domínio próprias do CNES, com dezenas/centenas de códigos granulares (e **zero-padded**: `TIPEQUIP` é `'01'`, não `'1'` — comparar sem o zero à esquerda casa silenciosamente 0 linhas, sem erro). **Não temos essas tabelas de domínio ingeridas no projeto.** As análises abaixo usam só o nível mais alto (`TIPEQUIP = '01'` "Diagnóstico por Imagem", `TP_LEITO = '1'` "Cirúrgico" — este não é zero-padded), que é razoavelmente estável — mas para distinguir precisamente "tomógrafo" de "raio-x" dentro de Diagnóstico por Imagem, por exemplo, é preciso buscar e ingerir o Dicionário de Dados do CNES (tabelas `tb_tipo_equipamento`/`tb_equipamento`) na camada silver.
- `MICR_REG` (microrregião) **parou de ser preenchido pelo DATASUS a partir da competência 202312 (dez/2023)** — em `cnes_lt` e `cnes_pf`, toda competência de 202312 em diante vem 100% vazia nesse campo; a última competência com o campo populado é **202311**. Isso não é um problema do pipeline de ingestão, é uma mudança na publicação da fonte. Qualquer análise por microrregião (seções 2.4 e 4.4) precisa fixar a competência em `'202311'` em vez de usar a mais recente — documentado e aplicado nas células correspondentes.

### Achado 4 — SISAB/SIAPS não está disponível em `data/raw`

Todo o bloco 3 (Atenção Primária) depende de indicadores do SISAB (cobertura de Estratégia Saúde da Família, desempenho da APS). Esse pipeline de ingestão ainda não existe no projeto — `internasus/ingestion/` só cobre CNES/SIA/SIH (via `pysus`) e população (via SIDRA). As células da seção 3 abaixo checam a existência da fonte e param com um aviso — não é bug de SQL, é dado que falta ingerir (API do SISAB em `https://sisab.saude.gov.br` ou extração manual).

## 1. Filas para especialistas, exames e cirurgias
**Objetivo:** Cruzar a demanda ambulatorial/hospitalar com a infraestrutura e serviços (SIA/SIH vs CNES).
- **1.1 / 1.2** Gargalos de exames diagnósticos e ociosidade vs sobrecarga de equipamentos de imagem
- **1.3** Taxa de Ocupação e Pressão sobre Leitos Cirúrgicos
- **1.4** Defasagem entre oferta de serviços especializados e demanda ambulatorial

In [15]:
# 1.1 / 1.2 — Gargalos de exames diagnósticos e ociosidade vs sobrecarga de equipamentos de imagem
gargalos_exames = conn.execute(f"""
    WITH producao_exames AS (
        SELECT LEFT(PA_MUNPCN, 6) AS cod_mun, COUNT(*) AS total_exames_realizados
        FROM sia
        WHERE PA_PROC_ID LIKE '02%' AND ano = {ANO_REF}  -- 02 = finalidade diagnóstica (SIGTAP)
        GROUP BY 1
    ),
    equipamentos_disp AS (
        SELECT CODUFMUN AS cod_mun, SUM(TRY_CAST(QT_EXIST AS INTEGER)) AS total_equipamentos
        FROM cnes_eq_atual
        WHERE TIPEQUIP = '01'  -- Diagnóstico por Imagem (zero-padded — ver Achado 3)
        GROUP BY 1
    )
    SELECT
        i.cod_mun,
        i.nome_mun,
        i.populacao,
        COALESCE(e.total_equipamentos, 0) AS equipamentos,
        COALESCE(p.total_exames_realizados, 0) AS exames_realizados,
        ROUND(COALESCE(p.total_exames_realizados, 0) * 1000.0 / NULLIF(i.populacao, 0), 2) AS exames_por_mil_hab,
        ROUND(COALESCE(p.total_exames_realizados, 0) / NULLIF(e.total_equipamentos, 0), 2) AS exames_por_equipamento,
        CASE
            WHEN COALESCE(e.total_equipamentos, 0) = 0 AND COALESCE(p.total_exames_realizados, 0) > 0 THEN 'Sem equipamento próprio (gargalo)'
            WHEN e.total_equipamentos > 0 AND COALESCE(p.total_exames_realizados, 0) = 0 THEN 'Equipamento ocioso'
            ELSE 'Com produção e equipamento'
        END AS situacao
    FROM ibge_pop_ref i
    LEFT JOIN equipamentos_disp e ON i.cod_mun = e.cod_mun
    LEFT JOIN producao_exames p ON i.cod_mun = p.cod_mun
    ORDER BY exames_por_equipamento DESC NULLS LAST
""").df()

display(gargalos_exames.head(10))
print()
print("Municípios com produção mas sem equipamento próprio (gargalo):", (gargalos_exames["situacao"] == "Sem equipamento próprio (gargalo)").sum())
print("Municípios com equipamento ocioso (zero exames no ano):", (gargalos_exames["situacao"] == "Equipamento ocioso").sum())

,cod_mun,nome_mun,populacao,equipamentos,exames_realizados,exames_por_mil_hab,exames_por_equipamento,situacao
0,350075,Alambari - SP,6373,1.0,2957,463.99,2957.00,Com produção e equipamento
1,354530,Salto de Pirapora - SP,45262,14.0,30898,682.65,2207.00,Com produção e equipamento
2,351970,Ibiúna - SP,77801,16.0,34885,448.39,2180.31,Com produção e equipamento
3,353790,Pilar do Sul - SP,28459,9.0,18227,640.47,2025.22,Com produção e equipamento
4,353780,Piedade - SP,54266,16.0,27986,515.72,1749.13,Com produção e equipamento
5,351880,Guarulhos - SP,1349100,521.0,809264,599.85,1553.29,Com produção e equipamento
6,352620,Juquitiba - SP,27969,6.0,8525,304.80,1420.83,Com produção e equipamento
7,351760,Guapiara - SP,17225,3.0,4109,238.55,1369.67,Com produção e equipamento
8,355350,Tapiraí - SP,8122,5.0,6373,784.66,1274.60,Com produção e equipamento
9,354995,São Lourenço da Serra - SP,16479,5.0,6295,382.00,1259.00,Com produção e equipamento



Municípios com produção mas sem equipamento próprio (gargalo): 28
Municípios com equipamento ocioso (zero exames no ano): 11


In [16]:
# 1.3 — Taxa de Ocupação e Pressão sobre Leitos Cirúrgicos
pressao_leitos = conn.execute(f"""
    WITH internacoes_cirurgicas AS (
        SELECT CNES, COUNT(*) AS volume_internacoes, SUM(TRY_CAST(DIAS_PERM AS INTEGER)) AS total_dias_permanencia
        FROM sih
        WHERE PROC_REA LIKE '04%' AND TRY_CAST(DIAS_PERM AS INTEGER) > 0 AND ano = {ANO_REF}  -- 04 = procedimentos cirúrgicos (SIGTAP)
        GROUP BY CNES
    ),
    leitos_cirurgicos AS (
        SELECT CNES, CODUFMUN AS cod_mun, SUM(TRY_CAST(QT_EXIST AS INTEGER)) AS qtd_leitos_cirurgicos
        FROM cnes_lt_atual
        WHERE TP_LEITO = '1'  -- Cirúrgico, conforme domínio CNES-LT (ver Achado 3)
        GROUP BY CNES, CODUFMUN
    )
    SELECT
        l.cod_mun,
        SUM(l.qtd_leitos_cirurgicos) AS total_leitos_cir,
        SUM(i.volume_internacoes) AS total_cirurgias,
        -- Estimativa de ocupação: (dias ocupados no ano) / (leitos * 365 dias) * 100
        ROUND((SUM(i.total_dias_permanencia) / NULLIF(SUM(l.qtd_leitos_cirurgicos), 0) / 365.0) * 100, 2) AS taxa_ocupacao_estimada_pct
    FROM leitos_cirurgicos l
    LEFT JOIN internacoes_cirurgicas i ON l.CNES = i.CNES
    GROUP BY l.cod_mun
    ORDER BY taxa_ocupacao_estimada_pct DESC NULLS LAST
""").df()

display(pressao_leitos.head(10))

,cod_mun,total_leitos_cir,total_cirurgias,taxa_ocupacao_estimada_pct
0,350920,1.0,508.0,317.53
1,355670,12.0,1965.0,177.24
2,353260,4.0,1352.0,154.93
3,355280,88.0,7890.0,111.46
4,355070,15.0,2365.0,110.58
5,350960,4.0,781.0,110.07
6,352210,34.0,4493.0,102.01
7,352230,44.0,3691.0,100.56
8,351640,94.0,4267.0,100.38
9,352310,71.0,5312.0,100.36


**Observação sobre 1.4:** não existe chave direta entre a classificação de serviços do CNES (`SERV_ESP`/`CLASS_SR`) e a produção do SIA — são vocabulários diferentes. Como proxy, comparamos o total de serviços especializados cadastrados por município (`cnes_sr_atual`) com o volume de produção ambulatorial (`sia`), ambos per capita. Uma tabela de correspondência `SERV_ESP` ↔ CBO/SIGTAP é necessária na camada silver para uma resposta por especialidade específica.

In [17]:
# 1.4 — Defasagem entre oferta de serviços especializados cadastrados e demanda ambulatorial (proxy agregado)
defasagem_especialidades = conn.execute(f"""
    WITH oferta AS (
        SELECT CODUFMUN AS cod_mun, COUNT(*) AS servicos_especializados_cadastrados
        FROM cnes_sr_atual GROUP BY 1
    ),
    demanda AS (
        SELECT LEFT(PA_UFMUN, 6) AS cod_mun, COUNT(*) AS producao_ambulatorial
        FROM sia WHERE ano = {ANO_REF} GROUP BY 1
    )
    SELECT
        i.cod_mun, i.nome_mun, i.populacao,
        COALESCE(o.servicos_especializados_cadastrados, 0) AS servicos_cadastrados,
        COALESCE(d.producao_ambulatorial, 0) AS producao_ambulatorial,
        ROUND(COALESCE(d.producao_ambulatorial, 0) * 1000.0 / NULLIF(i.populacao, 0), 2) AS producao_por_mil_hab,
        ROUND(COALESCE(o.servicos_especializados_cadastrados, 0) * 1000.0 / NULLIF(i.populacao, 0), 4) AS servicos_por_mil_hab
    FROM ibge_pop_ref i
    LEFT JOIN oferta o ON i.cod_mun = o.cod_mun
    LEFT JOIN demanda d ON i.cod_mun = d.cod_mun
    ORDER BY producao_por_mil_hab DESC NULLS LAST
""").df()

display(defasagem_especialidades.head(10))

,cod_mun,nome_mun,populacao,servicos_cadastrados,producao_ambulatorial,producao_por_mil_hab,servicos_por_mil_hab
0,355220,Sorocaba - SP,762172,1982,6776188,8890.63,2.6005
1,355030,São Paulo - SP,11904961,41881,42446274,3565.43,3.5179
2,351880,Guarulhos - SP,1349100,2625,4637094,3437.18,1.9457
3,350950,Campinas - SP,1187974,5450,459626,386.90,4.5876
4,353960,Planalto - SP,4451,38,0,0.00,8.5374
5,354340,Ribeirão Preto - SP,731639,2377,0,0.00,3.2489
6,354425,Rosana - SP,17434,145,0,0.00,8.3171
7,354530,Salto de Pirapora - SP,45262,146,0,0.00,3.2257
8,354540,Salto Grande - SP,9221,70,0,0.00,7.5914
9,354700,Santa Maria da Serra - SP,5308,30,0,0.00,5.6518


## 2. Desigualdade Regional de Acesso (Rotas e Fugas)
**Objetivo:** Mapear o deslocamento de pacientes e a sobrecarga de polos regionais. Avaliar municípios exportadores, polos recebedores e infraestrutura per capita por microrregião.

In [18]:
# 2.1 Municípios "Exportadores" de Pacientes (Fuga Assistencial)
municipios_exportadores = conn.execute(f"""
    SELECT
        MUNIC_RES AS municipio_origem,
        COUNT(*) AS total_internacoes_paciente,
        SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) AS evasao_hospitalar,
        ROUND((SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS pct_evasao
    FROM sih
    WHERE ano = {ANO_REF}
    GROUP BY MUNIC_RES
    HAVING COUNT(*) > 100  -- filtro estatístico mínimo
    ORDER BY pct_evasao DESC
""").df()

display(municipios_exportadores.head(10))

# 2.2 Polos Regionais Sobrecarregados por Demanda Externa
polos_sobrecarregados = conn.execute(f"""
    SELECT
        MUNIC_MOV AS polo_recebedor,
        COUNT(*) AS total_atendimentos,
        SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) AS pacientes_externos,
        ROUND((SUM(CASE WHEN MUNIC_RES != MUNIC_MOV THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS pct_pressao_externa
    FROM sih
    WHERE ano = {ANO_REF}
    GROUP BY MUNIC_MOV
    ORDER BY pacientes_externos DESC
""").df()

display(polos_sobrecarregados.head(10))

,municipio_origem,total_internacoes_paciente,evasao_hospitalar,pct_evasao
0,352610,1293,1293.0,100.0
1,351519,486,486.0,100.0
2,354650,366,366.0,100.0
3,353810,1187,1187.0,100.0
4,354805,480,480.0,100.0
5,355730,676,676.0,100.0
6,292740,239,239.0,100.0
7,352870,338,338.0,100.0
8,354270,536,536.0,100.0
9,352990,1458,1458.0,100.0


,polo_recebedor,total_atendimentos,pacientes_externos,pct_pressao_externa
0,355030,757113,134254.0,17.73
1,354980,96106,54220.0,56.42
2,354340,94083,45930.0,48.82
3,350950,88993,31582.0,35.49
4,355220,61951,28142.0,45.43
5,354140,44789,27796.0,62.06
6,350550,33158,21704.0,65.46
7,350600,41754,18415.0,44.10
8,354990,59671,18243.0,30.57
9,350750,30264,17605.0,58.17


In [19]:
# 2.3 Regiões com menor proporção de leitos, equipamentos e serviços especializados por habitante
infra_per_capita = conn.execute("""
    WITH leitos AS (
        SELECT CODUFMUN AS cod_mun, SUM(TRY_CAST(QT_EXIST AS INTEGER)) AS total_leitos
        FROM cnes_lt_atual GROUP BY 1
    ),
    equipamentos AS (
        SELECT CODUFMUN AS cod_mun, SUM(TRY_CAST(QT_EXIST AS INTEGER)) AS total_equipamentos
        FROM cnes_eq_atual GROUP BY 1
    ),
    servicos AS (
        SELECT CODUFMUN AS cod_mun, COUNT(*) AS total_servicos
        FROM cnes_sr_atual GROUP BY 1
    )
    SELECT
        i.cod_mun, i.nome_mun, i.populacao,
        COALESCE(l.total_leitos, 0) AS leitos,
        COALESCE(e.total_equipamentos, 0) AS equipamentos,
        COALESCE(s.total_servicos, 0) AS servicos,
        ROUND(COALESCE(l.total_leitos, 0) * 1000.0 / NULLIF(i.populacao, 0), 3) AS leitos_por_mil_hab,
        ROUND(COALESCE(e.total_equipamentos, 0) * 1000.0 / NULLIF(i.populacao, 0), 3) AS equipamentos_por_mil_hab,
        ROUND(COALESCE(s.total_servicos, 0) * 1000.0 / NULLIF(i.populacao, 0), 3) AS servicos_por_mil_hab
    FROM ibge_pop_ref i
    LEFT JOIN leitos l ON i.cod_mun = l.cod_mun
    LEFT JOIN equipamentos e ON i.cod_mun = e.cod_mun
    LEFT JOIN servicos s ON i.cod_mun = s.cod_mun
    WHERE i.populacao > 0
    ORDER BY leitos_por_mil_hab ASC
""").df()

display(infra_per_capita.head(10))

,cod_mun,nome_mun,populacao,leitos,equipamentos,servicos,leitos_por_mil_hab,equipamentos_por_mil_hab,servicos_por_mil_hab
0,354805,Santo Antônio do Aracanguá - SP,8593,0.0,21.0,78,0.0,2.444,9.077
1,355050,São Pedro do Turvo - SP,7333,0.0,37.0,40,0.0,5.046,5.455
2,351710,Glicério - SP,4158,0.0,31.0,28,0.0,7.456,6.734
3,350430,Avaí - SP,4503,0.0,30.0,59,0.0,6.662,13.102
4,352725,Lourdes - SP,1962,0.0,3.0,23,0.0,1.529,11.723
5,352760,Luís Antônio - SP,12564,0.0,246.0,67,0.0,19.580,5.333
6,353100,Monções - SP,1947,0.0,33.0,45,0.0,16.949,23.112
7,351800,Guarani d'Oeste - SP,1999,0.0,22.0,47,0.0,11.006,23.512
8,353284,Nova Canaã Paulista - SP,2055,0.0,67.0,32,0.0,32.603,15.572
9,353320,Nova Independência - SP,4847,0.0,17.0,40,0.0,3.507,8.253


In [20]:
# 2.4 Microrregiões com pior infraestrutura hospitalar (leitos per capita) x "desenvolvimento" populacional
# Proxy de desenvolvimento: variação da população estimada entre o primeiro e o último ano disponíveis no SIDRA
variacao_pop = conn.execute("""
    WITH extremos AS (
        SELECT cod_mun,
               MAX(CASE WHEN ano = (SELECT MIN(ano) FROM ibge_pop_sp) THEN populacao END) AS pop_inicial,
               MAX(CASE WHEN ano = (SELECT MAX(ano) FROM ibge_pop_sp) THEN populacao END) AS pop_final
        FROM ibge_pop_sp
        GROUP BY cod_mun
    )
    SELECT cod_mun,
           ROUND((pop_final - pop_inicial) * 100.0 / NULLIF(pop_inicial, 0), 2) AS variacao_pop_pct
    FROM extremos
""").df()
conn.register("ibge_pop_variacao", variacao_pop)

leitos_x_crescimento = conn.execute("""
    WITH leitos_mun AS (
        SELECT MICR_REG AS microrregiao, CODUFMUN AS cod_mun, SUM(TRY_CAST(QT_EXIST AS INTEGER)) AS leitos
        FROM cnes_lt_micro  -- competência fixa 202311: última com MICR_REG preenchido (ver Achado 3)
        WHERE MICR_REG <> ''
        GROUP BY 1, 2
    )
    SELECT
        lm.microrregiao,
        SUM(lm.leitos) AS total_leitos,
        SUM(i.populacao) AS populacao_total,
        ROUND(SUM(lm.leitos) * 1000.0 / NULLIF(SUM(i.populacao), 0), 3) AS leitos_por_mil_hab,
        ROUND(AVG(v.variacao_pop_pct), 2) AS variacao_pop_media_pct
    FROM leitos_mun lm
    JOIN ibge_pop_ref i ON lm.cod_mun = i.cod_mun
    LEFT JOIN ibge_pop_variacao v ON lm.cod_mun = v.cod_mun
    GROUP BY lm.microrregiao
    ORDER BY leitos_por_mil_hab ASC
""").df()

display(leitos_x_crescimento.head(10))
print("Correlação leitos/mil hab x variação populacional:",
      round(leitos_x_crescimento["leitos_por_mil_hab"].corr(leitos_x_crescimento["variacao_pop_media_pct"]), 3))

,microrregiao,total_leitos,populacao_total,leitos_por_mil_hab,variacao_pop_media_pct
0,043,2.0,11904961.0,0.000,-3.36
1,018,14.0,11904961.0,0.001,-3.36
2,005,6.0,11904961.0,0.001,-3.36
3,0030,15.0,11904961.0,0.001,-3.36
4,101,8.0,11904961.0,0.001,-3.36
5,0016,14.0,11904961.0,0.001,-3.36
6,0021,22.0,11904961.0,0.002,-3.36
7,0003,23.0,11904961.0,0.002,-3.36
8,0001,28.0,11904961.0,0.002,-3.36
9,70,19.0,11904961.0,0.002,-3.36


Correlação leitos/mil hab x variação populacional: 0.315


## 3. Problemas na Atenção Primária (UBS)
**Objetivo:** Avaliar a cobertura da Atenção Primária (APS) e sua correlação com Internações por Condições Sensíveis (ICSAP).

In [21]:
# 3.1 / 3.2 / 3.3 — dependem do SISAB/SIAPS (ver Achado 4)
tabelas = conn.execute("SHOW TABLES").df()["name"].tolist()

if "sisab_indicadores" not in tabelas:
    print(
        "[Bloco 3] Pulado: as 3 perguntas dependem do SISAB/SIAPS (cobertura de ESF, "
        "desempenho da APS), que não está disponível em data/raw (ver Achado 4).\n"
        "É necessário implementar a ingestão dessa fonte em internasus/ingestion/ antes de responder:\n"
        "  3.1 Internações evitáveis (ICSAP, via SIH.DIAG_PRINC) x cobertura de ESF (SISAB)\n"
        "  3.2 Capacidade das equipes de Saúde da Família (SISAB) x encaminhamentos de média/alta\n"
        "      complexidade (SIA.PA_NIVCPL)\n"
        "  3.3 Cobertura da APS (SISAB) x pressão hospitalar (SIH) x população (IBGE)\n"
    )
else:
    icsap_vs_aps = conn.execute(f"""
        WITH internacoes_csap AS (
            SELECT
                MUNIC_RES AS cod_mun,
                COUNT(*) AS total_internacoes,
                SUM(CASE WHEN SUBSTR(DIAG_PRINC, 1, 3) IN ('J45','I10','E10','E11','E14','A09') THEN 1 ELSE 0 END) AS internacoes_csap
            FROM sih
            WHERE ano = {ANO_REF}
            GROUP BY MUNIC_RES
        ),
        desempenho_aps AS (
            SELECT CODUFMUN AS cod_mun, AVG(cobertura_estrategia_saude_familia) AS pct_cobertura_esf
            FROM sisab_indicadores
            GROUP BY CODUFMUN
        )
        SELECT
            i.cod_mun, d.pct_cobertura_esf, i.total_internacoes, i.internacoes_csap,
            ROUND((i.internacoes_csap * 100.0) / NULLIF(i.total_internacoes, 0), 2) AS pct_internacoes_evitaveis
        FROM internacoes_csap i
        JOIN desempenho_aps d ON i.cod_mun = d.cod_mun
        ORDER BY pct_internacoes_evitaveis DESC
    """).df()
    display(icsap_vs_aps.head(10))

[Bloco 3] Pulado: as 3 perguntas dependem do SISAB/SIAPS (cobertura de ESF, desempenho da APS), que não está disponível em data/raw (ver Achado 4).
É necessário implementar a ingestão dessa fonte em internasus/ingestion/ antes de responder:
  3.1 Internações evitáveis (ICSAP, via SIH.DIAG_PRINC) x cobertura de ESF (SISAB)
  3.2 Capacidade das equipes de Saúde da Família (SISAB) x encaminhamentos de média/alta
      complexidade (SIA.PA_NIVCPL)
  3.3 Cobertura da APS (SISAB) x pressão hospitalar (SIH) x população (IBGE)



## 4. Falta ou Má Distribuição de Profissionais Especializados
**Objetivo:** Cruzar CBOs (Profissionais) com a infraestrutura ociosa e alta complexidade (identificar vazios assistenciais).

In [22]:
# 4.1 Vazios Assistenciais: médicos e enfermeiros por 1.000 habitantes
vazios_assistenciais = conn.execute("""
    WITH total_profissionais AS (
        SELECT
            CODUFMUN AS cod_mun,
            COUNT(DISTINCT CASE WHEN CBO LIKE '225%' THEN CPF_PROF END) AS qtd_medicos,
            COUNT(DISTINCT CASE WHEN CBO LIKE '2235%' THEN CPF_PROF END) AS qtd_enfermeiros
        FROM cnes_pf_atual
        GROUP BY 1
    )
    SELECT
        i.cod_mun, i.nome_mun, i.populacao,
        COALESCE(p.qtd_medicos, 0) AS medicos,
        COALESCE(p.qtd_enfermeiros, 0) AS enfermeiros,
        ROUND(COALESCE(p.qtd_medicos, 0) * 1000.0 / NULLIF(i.populacao, 0), 3) AS medicos_por_mil_hab,
        ROUND(COALESCE(p.qtd_enfermeiros, 0) * 1000.0 / NULLIF(i.populacao, 0), 3) AS enf_por_mil_hab
    FROM ibge_pop_ref i
    LEFT JOIN total_profissionais p ON i.cod_mun = p.cod_mun
    WHERE i.populacao > 0
    ORDER BY medicos_por_mil_hab ASC
""").df()

display(vazios_assistenciais.head(10))

,cod_mun,nome_mun,populacao,medicos,enfermeiros,medicos_por_mil_hab,enf_por_mil_hab
0,353680,Pedra Bela - SP,6745,6,7,0.890,1.038
1,350770,Braúna - SP,5477,5,7,0.913,1.278
2,355490,Três Fronteiras - SP,7060,7,4,0.992,0.567
3,353600,Parapuã - SP,10720,11,16,1.026,1.493
4,353040,Mirassolândia - SP,4783,5,9,1.045,1.882
5,350910,Caiuá - SP,5599,6,8,1.072,1.429
6,355610,Valentim Gentil - SP,14634,16,19,1.093,1.298
7,354300,Ribeirão Branco - SP,18942,21,20,1.109,1.056
8,353830,Piquerobi - SP,3287,4,4,1.217,1.217
9,352780,Lupércio - SP,4004,5,7,1.249,1.748


In [23]:
# 4.2 Alocação de especialistas x produção de alta complexidade (SIA.PA_NIVCPL='3') e cirurgias (SIH)
especialistas_por_mun = conn.execute("""
    SELECT CODUFMUN AS cod_mun, COUNT(DISTINCT CPF_PROF) AS especialistas
    FROM cnes_pf_atual
    WHERE CBO LIKE '225%'
    GROUP BY 1
""").df()

producao_alta_complexidade = conn.execute(f"""
    SELECT LEFT(PA_UFMUN, 6) AS cod_mun, COUNT(*) AS producao_alta_complexidade
    FROM sia
    WHERE PA_NIVCPL = '3' AND ano = {ANO_REF}  -- 3 = alta complexidade (domínio PA_NIVCPL do SIA)
    GROUP BY 1
""").df()

cirurgias_sih = conn.execute(f"""
    SELECT MUNIC_MOV AS cod_mun, COUNT(*) AS cirurgias
    FROM sih WHERE PROC_REA LIKE '04%' AND ano = {ANO_REF} GROUP BY 1
""").df()

alocacao_especialistas = (
    especialistas_por_mun
    .merge(producao_alta_complexidade, on="cod_mun", how="outer")
    .merge(cirurgias_sih, on="cod_mun", how="outer")
    .fillna(0)
)
alocacao_especialistas["demanda_alta_complexidade"] = (
    alocacao_especialistas["producao_alta_complexidade"] + alocacao_especialistas["cirurgias"]
)
alocacao_especialistas["demanda_por_especialista"] = (
    alocacao_especialistas["demanda_alta_complexidade"] / alocacao_especialistas["especialistas"].replace(0, pd.NA)
).round(2)

display(alocacao_especialistas.sort_values("demanda_por_especialista", ascending=False).head(10))

,cod_mun,especialistas,producao_alta_complexidade,cirurgias,demanda_alta_complexidade,demanda_por_especialista
581,355220,5339,1290140.0,31131.0,1321271.0,247.48
562,355030,74074,8056038.0,357411.0,8413449.0,113.58
212,351880,5927,363792.0,24840.0,388632.0,65.57
48,350420,49,0.0,2112.0,2112.0,43.10
365,353260,41,0.0,1353.0,1353.0,33.00
154,351390,142,0.0,4309.0,4309.0,30.35
98,350870,49,0.0,1138.0,1138.0,23.22
27,350250,128,0.0,2952.0,2952.0,23.06
496,354425,58,0.0,1287.0,1287.0,22.19
406,353620,233,0.0,5146.0,5146.0,22.09


In [24]:
# 4.3 Infraestrutura de imagem adequada, mas carente de médicos especialistas
infra_sem_profissionais = conn.execute("""
    WITH medicos_especialistas AS (
        SELECT CNES, COUNT(DISTINCT CPF_PROF) AS qtd_especialistas
        FROM cnes_pf_atual
        WHERE CBO LIKE '225%'
        GROUP BY CNES
    ),
    equipamentos_alta_comp AS (
        SELECT CNES, CODUFMUN AS cod_mun, SUM(TRY_CAST(QT_EXIST AS INTEGER)) AS qtd_equip
        FROM cnes_eq_atual
        WHERE TIPEQUIP = '01'  -- Diagnóstico por Imagem (zero-padded — ver Achado 3)
        GROUP BY CNES, CODUFMUN
    )
    SELECT
        eq.cod_mun, eq.CNES, eq.qtd_equip,
        COALESCE(pf.qtd_especialistas, 0) AS especialistas_vinculados,
        CASE
            WHEN COALESCE(pf.qtd_especialistas, 0) = 0 AND eq.qtd_equip > 0 THEN 'Infra Ociosa (Falta RH)'
            ELSE 'Operacional'
        END AS status_capacidade
    FROM equipamentos_alta_comp eq
    LEFT JOIN medicos_especialistas pf ON eq.CNES = pf.CNES
    WHERE eq.qtd_equip > 0
    ORDER BY especialistas_vinculados ASC, eq.qtd_equip DESC
""").df()

display(infra_sem_profissionais.head(10))
print("Estabelecimentos com infraestrutura ociosa por falta de RH:",
      (infra_sem_profissionais["status_capacidade"] == "Infra Ociosa (Falta RH)").sum())

,cod_mun,CNES,qtd_equip,especialistas_vinculados,status_capacidade
0,351907,5811090,135.0,0,Infra Ociosa (Falta RH)
1,353440,7252536,101.0,0,Infra Ociosa (Falta RH)
2,350710,3597512,101.0,0,Infra Ociosa (Falta RH)
3,350950,0976148,39.0,0,Infra Ociosa (Falta RH)
4,350320,2747286,37.0,0,Infra Ociosa (Falta RH)
5,354340,2055228,33.0,0,Infra Ociosa (Falta RH)
6,350280,2030020,31.0,0,Infra Ociosa (Falta RH)
7,350760,0906026,19.0,0,Infra Ociosa (Falta RH)
8,353060,7982275,14.0,0,Infra Ociosa (Falta RH)
9,350600,5918545,14.0,0,Infra Ociosa (Falta RH)


Estabelecimentos com infraestrutura ociosa por falta de RH: 11833


**Observação sobre 4.4:** nem o CNES nem a base de população do IBGE/SIDRA ingerida trazem a mesorregião do município — só há `MICR_REG` (microrregião) no CNES. Usamos microrregião como proxy; para responder com mesorregião de fato é preciso ingerir a malha territorial do IBGE (Divisão Territorial Brasileira — DTB) e mapear município → mesorregião na camada silver.

In [25]:
# 4.4 Distribuição espacial de especialidades médicas por microrregião (proxy de mesorregião)
distribuicao_especialidades = conn.execute("""
    SELECT MICR_REG AS microrregiao, SUBSTR(CBO, 1, 4) AS especialidade_cbo, COUNT(DISTINCT CPF_PROF) AS profissionais
    FROM cnes_pf_micro  -- competência fixa 202311: última com MICR_REG preenchido (ver Achado 3)
    WHERE CBO LIKE '225%' AND MICR_REG <> ''
    GROUP BY 1, 2
""").df()

especialidade_dominante_por_microrregiao = (
    distribuicao_especialidades
    .sort_values("profissionais", ascending=False)
    .groupby("microrregiao")
    .head(1)
    .sort_values("profissionais")
)

display(especialidade_dominante_por_microrregiao.head(10))

,microrregiao,especialidade_cbo,profissionais
84,AVARE,2251,1
145,209,2252,1
152,217,2251,1
87,8,2252,1
70,204,2252,1
89,VII,2251,1
69,I,2253,1
144,FRANCA,2251,1
156,XV,2251,1
127,6,2251,1
